# 🏠 European Airbnb Data Engineering Project

## 📌 Project Overview

This project presents an end-to-end data engineering pipeline for analyzing Airbnb listings across major European cities.

The workflow starts with raw Airbnb datasets, then progresses through data integration, data cleaning, web scraping, feature engineering, SQL Server data warehousing, and finally an interactive Power BI dashboard for business intelligence.

---

## 🎯 Project Objectives

- Build a unified master dataset from multiple European Airbnb datasets.
- Clean and standardize the raw data.
- Enrich the dataset using additional Airbnb information collected through web scraping.
- Store the final analytical dataset in SQL Server.
- Develop an interactive Power BI dashboard to generate business insights.

In [117]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)

## 🛠️ Technologies Used

| Technology | Purpose |
|------------|---------|
| Python | Data Processing & Automation |
| Pandas | Data Cleaning & Analysis |
| Selenium | Web Scraping |
| SQL Server | Data Warehouse |
| Power BI | Dashboard & Data Visualization |
| Jupyter Notebook | Documentation & Workflow |

# 📂 Load Raw Airbnb Dataset

The project begins by loading the raw Airbnb datasets collected from multiple European cities.

Each city contains two datasets:

- Weekdays Listings
- Weekend Listings

These datasets will later be combined into one unified master dataset.

In [118]:
bronze_path = Path("../01_bronze_layer")

csv_files = list(bronze_path.glob("*.csv"))

print(f"Number of datasets: {len(csv_files)}")

Number of datasets: 20


In [119]:
csv_files

[WindowsPath('../01_bronze_layer/amsterdam_weekdays.csv'),
 WindowsPath('../01_bronze_layer/amsterdam_weekends.csv'),
 WindowsPath('../01_bronze_layer/athens_weekdays.csv'),
 WindowsPath('../01_bronze_layer/athens_weekends.csv'),
 WindowsPath('../01_bronze_layer/barcelona_weekdays.csv'),
 WindowsPath('../01_bronze_layer/barcelona_weekends.csv'),
 WindowsPath('../01_bronze_layer/berlin_weekdays.csv'),
 WindowsPath('../01_bronze_layer/berlin_weekends.csv'),
 WindowsPath('../01_bronze_layer/budapest_weekdays.csv'),
 WindowsPath('../01_bronze_layer/budapest_weekends.csv'),
 WindowsPath('../01_bronze_layer/lisbon_weekdays.csv'),
 WindowsPath('../01_bronze_layer/lisbon_weekends.csv'),
 WindowsPath('../01_bronze_layer/london_weekdays.csv'),
 WindowsPath('../01_bronze_layer/london_weekends.csv'),
 WindowsPath('../01_bronze_layer/paris_weekdays.csv'),
 WindowsPath('../01_bronze_layer/paris_weekends.csv'),
 WindowsPath('../01_bronze_layer/rome_weekdays.csv'),
 WindowsPath('../01_bronze_layer/rom

## 📊 Initial Dataset Exploration

Before building the master dataset, it is important to verify that all raw datasets are successfully loaded.

In [120]:
for file in csv_files:
    print(file.name)

amsterdam_weekdays.csv
amsterdam_weekends.csv
athens_weekdays.csv
athens_weekends.csv
barcelona_weekdays.csv
barcelona_weekends.csv
berlin_weekdays.csv
berlin_weekends.csv
budapest_weekdays.csv
budapest_weekends.csv
lisbon_weekdays.csv
lisbon_weekends.csv
london_weekdays.csv
london_weekends.csv
paris_weekdays.csv
paris_weekends.csv
rome_weekdays.csv
rome_weekends.csv
vienna_weekdays.csv
vienna_weekends.csv


In [121]:
len(csv_files)

20

# 📂 Building the Master Dataset

The raw Airbnb datasets are distributed across multiple CSV files, with each file representing a specific city and day type (weekdays or weekends).

To simplify data processing, all datasets are merged into a single master dataset while preserving important metadata such as the city name and day type.

This unified dataset serves as the foundation for the subsequent data cleaning and feature engineering stages.

In [122]:
from pathlib import Path
import pandas as pd

bronze_folder = Path("../01_bronze_layer")

csv_files = list(bronze_folder.glob("*.csv"))

df_list = []

for file in csv_files:
    filename = file.stem
    city, day_type = filename.rsplit("_", 1)

    temp_df = pd.read_csv(file)
    temp_df["city"] = city.capitalize()
    temp_df["day_type"] = day_type

    df_list.append(temp_df)

master_df = pd.concat(df_list, ignore_index=True)
master_df.head()

,Unnamed: 0,realSum,room_type,room_shared,room_private,person_capacity,host_is_superhost,multi,biz,cleanliness_rating,guest_satisfaction_overall,bedrooms,dist,metro_dist,attr_index,attr_index_norm,rest_index,rest_index_norm,lng,lat,city,day_type
0,0,194.033698,Private room,False,True,2.0,False,1,0,10.0,93.0,1,5.022964,2.539380,78.690379,4.166708,98.253896,6.846473,4.90569,52.41772,Amsterdam,weekdays
1,1,344.245776,Private room,False,True,4.0,False,0,0,8.0,85.0,1,0.488389,0.239404,631.176378,33.421209,837.280757,58.342928,4.90005,52.37432,Amsterdam,weekdays
2,2,264.101422,Private room,False,True,2.0,False,0,1,9.0,87.0,1,5.748312,3.651621,75.275877,3.985908,95.386955,6.646700,4.97512,52.36103,Amsterdam,weekdays
3,3,433.529398,Private room,False,True,4.0,False,0,1,9.0,90.0,2,0.384862,0.439876,493.272534,26.119108,875.033098,60.973565,4.89417,52.37663,Amsterdam,weekdays
4,4,485.552926,Private room,False,True,2.0,True,0,0,10.0,98.0,1,0.544738,0.318693,552.830324,29.272733,815.305740,56.811677,4.90051,52.37508,Amsterdam,weekdays


In [123]:
master_df.shape

(51707, 22)

## ✅ Building Master Dataset Summary

All 20 raw Airbnb datasets were successfully merged into a single master dataset.

The resulting dataset contains:

- Listings from all available European cities.
- City information.
- Day type (Weekdays / Weekends).

The output of this stage is:

`master_dataset_raw.csv`

# 🧹 Data Cleaning

After building the master dataset, the next step is to clean and standardize the data before performing feature engineering.

The cleaning process includes:

- Removing unnecessary columns
- Renaming columns with meaningful names
- Converting data types
- Reordering columns
- Removing duplicate records
- Checking missing values

The output of this stage is the **master_dataset_clean.csv** dataset.

In [124]:
clean_df = pd.read_csv("../02_silver_layer/master_dataset_raw.csv")

clean_df.head()

,Unnamed: 0,realSum,room_type,room_shared,room_private,person_capacity,host_is_superhost,multi,biz,cleanliness_rating,guest_satisfaction_overall,bedrooms,dist,metro_dist,attr_index,attr_index_norm,rest_index,rest_index_norm,lng,lat,city,day_type
0,0,194.033698,Private room,False,True,2.0,False,1,0,10.0,93.0,1,5.022964,2.539380,78.690379,4.166708,98.253896,6.846473,4.90569,52.41772,Amsterdam,weekdays
1,1,344.245776,Private room,False,True,4.0,False,0,0,8.0,85.0,1,0.488389,0.239404,631.176378,33.421209,837.280757,58.342928,4.90005,52.37432,Amsterdam,weekdays
2,2,264.101422,Private room,False,True,2.0,False,0,1,9.0,87.0,1,5.748312,3.651621,75.275877,3.985908,95.386955,6.646700,4.97512,52.36103,Amsterdam,weekdays
3,3,433.529398,Private room,False,True,4.0,False,0,1,9.0,90.0,2,0.384862,0.439876,493.272534,26.119108,875.033098,60.973565,4.89417,52.37663,Amsterdam,weekdays
4,4,485.552926,Private room,False,True,2.0,True,0,0,10.0,98.0,1,0.544738,0.318693,552.830324,29.272733,815.305740,56.811677,4.90051,52.37508,Amsterdam,weekdays


In [125]:
clean_df.shape
clean_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 51707 entries, 0 to 51706
Data columns (total 22 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Unnamed: 0                  51707 non-null  int64  
 1   realSum                     51707 non-null  float64
 2   room_type                   51707 non-null  str    
 3   room_shared                 51707 non-null  bool   
 4   room_private                51707 non-null  bool   
 5   person_capacity             51707 non-null  float64
 6   host_is_superhost           51707 non-null  bool   
 7   multi                       51707 non-null  int64  
 8   biz                         51707 non-null  int64  
 9   cleanliness_rating          51707 non-null  float64
 10  guest_satisfaction_overall  51707 non-null  float64
 11  bedrooms                    51707 non-null  int64  
 12  dist                        51707 non-null  float64
 13  metro_dist                  51707 non-null

### Remove Unnecessary Columns

The dataset contains a few columns that are either redundant or not required for further analysis. These columns are removed to simplify the dataset.

In [126]:
clean_df.drop(columns=["Unnamed: 0"], inplace=True, errors="ignore")

clean_df.drop(columns=["room_shared", "room_private"], inplace=True)

clean_df.head()

,realSum,room_type,person_capacity,host_is_superhost,multi,biz,cleanliness_rating,guest_satisfaction_overall,bedrooms,dist,metro_dist,attr_index,attr_index_norm,rest_index,rest_index_norm,lng,lat,city,day_type
0,194.033698,Private room,2.0,False,1,0,10.0,93.0,1,5.022964,2.539380,78.690379,4.166708,98.253896,6.846473,4.90569,52.41772,Amsterdam,weekdays
1,344.245776,Private room,4.0,False,0,0,8.0,85.0,1,0.488389,0.239404,631.176378,33.421209,837.280757,58.342928,4.90005,52.37432,Amsterdam,weekdays
2,264.101422,Private room,2.0,False,0,1,9.0,87.0,1,5.748312,3.651621,75.275877,3.985908,95.386955,6.646700,4.97512,52.36103,Amsterdam,weekdays
3,433.529398,Private room,4.0,False,0,1,9.0,90.0,2,0.384862,0.439876,493.272534,26.119108,875.033098,60.973565,4.89417,52.37663,Amsterdam,weekdays
4,485.552926,Private room,2.0,True,0,0,10.0,98.0,1,0.544738,0.318693,552.830324,29.272733,815.305740,56.811677,4.90051,52.37508,Amsterdam,weekdays


### Rename Columns

To improve readability and maintain a consistent naming convention, several columns are renamed.

In [127]:
clean_df.rename(columns={
    "realSum": "price",
    "person_capacity": "guest_capacity",
    "host_is_superhost": "is_superhost",
    "multi": "multi_listing",
    "biz": "business_listing",
    "cleanliness_rating": "cleanliness_score",
    "guest_satisfaction_overall": "guest_satisfaction_score",
    "bedrooms": "bedroom_count",
    "dist": "distance_to_city_center",
    "metro_dist": "distance_to_metro",
    "attr_index": "attraction_index",
    "attr_index_norm": "normalized_attraction_index",
    "rest_index": "restaurant_index",
    "rest_index_norm": "normalized_restaurant_index",
    "lng": "longitude",
    "lat": "latitude"
}, inplace=True)

clean_df.head()

,price,room_type,guest_capacity,is_superhost,multi_listing,business_listing,cleanliness_score,guest_satisfaction_score,bedroom_count,distance_to_city_center,distance_to_metro,attraction_index,normalized_attraction_index,restaurant_index,normalized_restaurant_index,longitude,latitude,city,day_type
0,194.033698,Private room,2.0,False,1,0,10.0,93.0,1,5.022964,2.539380,78.690379,4.166708,98.253896,6.846473,4.90569,52.41772,Amsterdam,weekdays
1,344.245776,Private room,4.0,False,0,0,8.0,85.0,1,0.488389,0.239404,631.176378,33.421209,837.280757,58.342928,4.90005,52.37432,Amsterdam,weekdays
2,264.101422,Private room,2.0,False,0,1,9.0,87.0,1,5.748312,3.651621,75.275877,3.985908,95.386955,6.646700,4.97512,52.36103,Amsterdam,weekdays
3,433.529398,Private room,4.0,False,0,1,9.0,90.0,2,0.384862,0.439876,493.272534,26.119108,875.033098,60.973565,4.89417,52.37663,Amsterdam,weekdays
4,485.552926,Private room,2.0,True,0,0,10.0,98.0,1,0.544738,0.318693,552.830324,29.272733,815.305740,56.811677,4.90051,52.37508,Amsterdam,weekdays


### Reorder Columns

The dataset columns are reordered to improve readability, placing the most important descriptive fields at the beginning.

In [128]:
column_order = [
    "city",
    "day_type",
    "price",
    "room_type",
    "guest_capacity",
    "bedroom_count",
    "is_superhost",
    "multi_listing",
    "business_listing",
    "cleanliness_score",
    "guest_satisfaction_score",
    "distance_to_city_center",
    "distance_to_metro",
    "attraction_index",
    "normalized_attraction_index",
    "restaurant_index",
    "normalized_restaurant_index",
    "longitude",
    "latitude"
]

clean_df = clean_df[column_order]

clean_df.head()

,city,day_type,price,room_type,guest_capacity,bedroom_count,is_superhost,multi_listing,business_listing,cleanliness_score,guest_satisfaction_score,distance_to_city_center,distance_to_metro,attraction_index,normalized_attraction_index,restaurant_index,normalized_restaurant_index,longitude,latitude
0,Amsterdam,weekdays,194.033698,Private room,2.0,1,False,1,0,10.0,93.0,5.022964,2.539380,78.690379,4.166708,98.253896,6.846473,4.90569,52.41772
1,Amsterdam,weekdays,344.245776,Private room,4.0,1,False,0,0,8.0,85.0,0.488389,0.239404,631.176378,33.421209,837.280757,58.342928,4.90005,52.37432
2,Amsterdam,weekdays,264.101422,Private room,2.0,1,False,0,1,9.0,87.0,5.748312,3.651621,75.275877,3.985908,95.386955,6.646700,4.97512,52.36103
3,Amsterdam,weekdays,433.529398,Private room,4.0,2,False,0,1,9.0,90.0,0.384862,0.439876,493.272534,26.119108,875.033098,60.973565,4.89417,52.37663
4,Amsterdam,weekdays,485.552926,Private room,2.0,1,True,0,0,10.0,98.0,0.544738,0.318693,552.830324,29.272733,815.305740,56.811677,4.90051,52.37508


### Data Validation

The final step is to validate the dataset by checking duplicate records and missing values.

In [129]:
duplicates = clean_df.duplicated().sum()

print(f"Duplicate rows: {duplicates}")

clean_df.isnull().sum()

clean_df.shape

clean_df.to_csv(
    "../02_silver_layer/master_dataset_clean.csv",
    index=False
)

print("Clean dataset saved successfully.")

Duplicate rows: 0
Clean dataset saved successfully.


## ✅ Data Cleaning Summary

The master dataset has been successfully cleaned and standardized.

Completed tasks:

- Removed unnecessary columns.
- Renamed columns using descriptive names.
- Converted data types.
- Reordered columns for better readability.
- Removed duplicate records.
- Verified that no missing values exist.

The cleaned dataset is now ready for the web scraping and feature engineering stages.

# 🌐 Web Scraping

The original Airbnb dataset contains property information such as price, location, and guest satisfaction.

However, additional information was required to perform deeper analysis.

To enrich the dataset, Selenium was used to scrape Airbnb listings and collect additional city-level information that was not available in the original dataset.

### 📌 Extract Listing Information

For each Airbnb listing, the scraper extracts the following information:

- Property Name
- Listing Price
- Guest Rating
- Number of Reviews
- Superhost Status
- Guest Favorite Status
- Listing URL

## ⚙️ Scraping Workflow

The scraping pipeline follows these steps:

1. Read the cleaned Airbnb dataset.
2. Extract the available cities.
3. Search Airbnb for each city.
4. Collect listing URLs.
5. Visit every listing page.
6. Extract additional listing information.
7. Save the scraped data.

In [130]:
import pandas as pd
import time
import random

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

### 🌍 Target Cities

The scraper collects Airbnb listings from the following European cities:

In [131]:
cities = [
    "Amsterdam",
    "Athens",
    "Barcelona",
    "Berlin",
    "Budapest",
    "Lisbon",
    "London",
    "Paris",
    "Rome",
    "Vienna"
]

cities

['Amsterdam',
 'Athens',
 'Barcelona',
 'Berlin',
 'Budapest',
 'Lisbon',
 'London',
 'Paris',
 'Rome',
 'Vienna']

### 🔎 Search Airbnb

For each city, the scraper searches Airbnb using the following filters:

- Entire home/apartment
- Minimum 2 bedrooms

In [132]:
city = "Amsterdam"

url = f"https://www.airbnb.com/s/{city}/homes?room_types[]=Entire%20home%2Fapt&min_bedrooms=2"

print(url)

https://www.airbnb.com/s/Amsterdam/homes?room_types[]=Entire%20home%2Fapt&min_bedrooms=2


### 📌 Extract Listing Information

For each Airbnb listing, the scraper extracts the following information:

- Property Name
- Listing Price
- Guest Rating
- Number of Reviews
- Superhost Status
- Guest Favorite Status
- Listing URL

In [133]:
scraped_df = pd.read_csv("../02_silver_layer/airbnb_detailed_500_apartments.csv")

scraped_df.head()

,City,Name,Price,Rating,Reviews_Count,Superhost,Guest_Favorite,Link
0,Amsterdam,Stylish appartement | 10 Min to Amsterdam Central,61105,0.0,0,Yes,No,https://www.airbnb.com/rooms/17209362756958431...
1,Athens,Luxury Apartment in heart of Athens-Blue Graphite,64000,0.0,0,Yes,Yes,https://www.airbnb.com/rooms/10165895789719919...
2,Athens,1930s Architect-Designed Gem | 3BR 110m² Athens,15805,0.0,0,Yes,No,https://www.airbnb.com/rooms/17070383685257470...
3,Athens,Loft by the Sea,37855,0.0,0,Yes,No,https://www.airbnb.com/rooms/17211057955624048...
4,Athens,New 2BD/2BA Apt • Balcony • Prime & Quiet Loca...,41211,0.0,0,Yes,Yes,https://www.airbnb.com/rooms/10970736952302680...


In [134]:
scraped_df.shape

scraped_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 396 entries, 0 to 395
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   City            396 non-null    str    
 1   Name            396 non-null    str    
 2   Price           396 non-null    int64  
 3   Rating          396 non-null    float64
 4   Reviews_Count   396 non-null    int64  
 5   Superhost       396 non-null    str    
 6   Guest_Favorite  396 non-null    str    
 7   Link            396 non-null    str    
dtypes: float64(1), int64(2), str(5)
memory usage: 24.9 KB


## ✅ Web Scraping Summary

The scraper successfully collected detailed Airbnb listing information from multiple European cities.

The extracted dataset contains:

- Property Name
- Price
- Rating
- Number of Reviews
- Superhost Status
- Guest Favorite Status
- Listing URL

The resulting dataset will be used in the next stage to build a unified Airbnb dataset.

# ⚙️ Feature Engineering

The scraped Airbnb dataset contains additional listing information that is not available in the original dataset.

In this stage, the scraped data is cleaned, transformed, and merged with the master Airbnb dataset to create a richer dataset for analysis and visualization.

In [135]:
master_df = pd.read_csv("../02_silver_layer/master_dataset_clean.csv")

scraped_df = pd.read_csv("../02_silver_layer/airbnb_detailed_500_apartments.csv")

In [136]:
print(master_df.shape)
print(scraped_df.shape)

(51707, 19)
(396, 8)


In [137]:
master_df.head()

,city,day_type,price,room_type,guest_capacity,bedroom_count,is_superhost,multi_listing,business_listing,cleanliness_score,guest_satisfaction_score,distance_to_city_center,distance_to_metro,attraction_index,normalized_attraction_index,restaurant_index,normalized_restaurant_index,longitude,latitude
0,Amsterdam,weekdays,194.033698,Private room,2.0,1,False,1,0,10.0,93.0,5.022964,2.539380,78.690379,4.166708,98.253896,6.846473,4.90569,52.41772
1,Amsterdam,weekdays,344.245776,Private room,4.0,1,False,0,0,8.0,85.0,0.488389,0.239404,631.176378,33.421209,837.280757,58.342928,4.90005,52.37432
2,Amsterdam,weekdays,264.101422,Private room,2.0,1,False,0,1,9.0,87.0,5.748312,3.651621,75.275877,3.985908,95.386955,6.646700,4.97512,52.36103
3,Amsterdam,weekdays,433.529398,Private room,4.0,2,False,0,1,9.0,90.0,0.384862,0.439876,493.272534,26.119108,875.033098,60.973565,4.89417,52.37663
4,Amsterdam,weekdays,485.552926,Private room,2.0,1,True,0,0,10.0,98.0,0.544738,0.318693,552.830324,29.272733,815.305740,56.811677,4.90051,52.37508


In [138]:
scraped_df.head()

,City,Name,Price,Rating,Reviews_Count,Superhost,Guest_Favorite,Link
0,Amsterdam,Stylish appartement | 10 Min to Amsterdam Central,61105,0.0,0,Yes,No,https://www.airbnb.com/rooms/17209362756958431...
1,Athens,Luxury Apartment in heart of Athens-Blue Graphite,64000,0.0,0,Yes,Yes,https://www.airbnb.com/rooms/10165895789719919...
2,Athens,1930s Architect-Designed Gem | 3BR 110m² Athens,15805,0.0,0,Yes,No,https://www.airbnb.com/rooms/17070383685257470...
3,Athens,Loft by the Sea,37855,0.0,0,Yes,No,https://www.airbnb.com/rooms/17211057955624048...
4,Athens,New 2BD/2BA Apt • Balcony • Prime & Quiet Loca...,41211,0.0,0,Yes,Yes,https://www.airbnb.com/rooms/10970736952302680...


## 📊 Explore the Scraped Dataset

Before merging the datasets, the scraped data is inspected to understand its structure and identify any missing values.

In [139]:
scraped_df.info()
scraped_df.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 396 entries, 0 to 395
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   City            396 non-null    str    
 1   Name            396 non-null    str    
 2   Price           396 non-null    int64  
 3   Rating          396 non-null    float64
 4   Reviews_Count   396 non-null    int64  
 5   Superhost       396 non-null    str    
 6   Guest_Favorite  396 non-null    str    
 7   Link            396 non-null    str    
dtypes: float64(1), int64(2), str(5)
memory usage: 24.9 KB


City              0
Name              0
Price             0
Rating            0
Reviews_Count     0
Superhost         0
Guest_Favorite    0
Link              0
dtype: int64

## 🧹 Handle Missing Values

In [140]:
scraped_df = scraped_df.dropna(subset=["Name", "Price"])

In [141]:
scraped_df["Rating"] = scraped_df["Rating"].fillna("Not Rated")

In [142]:
scraped_df.isnull().sum()

City              0
Name              0
Price             0
Rating            0
Reviews_Count     0
Superhost         0
Guest_Favorite    0
Link              0
dtype: int64

In [143]:
scraped_df.head()

,City,Name,Price,Rating,Reviews_Count,Superhost,Guest_Favorite,Link
0,Amsterdam,Stylish appartement | 10 Min to Amsterdam Central,61105,0.0,0,Yes,No,https://www.airbnb.com/rooms/17209362756958431...
1,Athens,Luxury Apartment in heart of Athens-Blue Graphite,64000,0.0,0,Yes,Yes,https://www.airbnb.com/rooms/10165895789719919...
2,Athens,1930s Architect-Designed Gem | 3BR 110m² Athens,15805,0.0,0,Yes,No,https://www.airbnb.com/rooms/17070383685257470...
3,Athens,Loft by the Sea,37855,0.0,0,Yes,No,https://www.airbnb.com/rooms/17211057955624048...
4,Athens,New 2BD/2BA Apt • Balcony • Prime & Quiet Loca...,41211,0.0,0,Yes,Yes,https://www.airbnb.com/rooms/10970736952302680...


## 💰 Clean Price Column

In [144]:
scraped_df["Price"] = (
    scraped_df["Price"]
    .str.replace("ج.م", "", regex=False)
    .str.replace("م؟ج", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

scraped_df["Price"] = pd.to_numeric(scraped_df["Price"], errors="coerce")

AttributeError: Can only use .str accessor with string values, not integer

In [ ]:
scraped_df.head()

,City,Name,Price,Rating,Reviews_Count,Superhost,Guest_Favorite,Link
0,Amsterdam,Stylish appartement | 10 Min to Amsterdam Central,61105,Not Rated,0,Yes,No,https://www.airbnb.com/rooms/17209362756958431...
1,Athens,Luxury Apartment in heart of Athens-Blue Graphite,64000,How reviews work,0,Yes,Yes,https://www.airbnb.com/rooms/10165895789719919...
2,Athens,1930s Architect-Designed Gem | 3BR 110m² Athens,15805,How reviews work,0,Yes,No,https://www.airbnb.com/rooms/17070383685257470...
3,Athens,Loft by the Sea,37855,Not Rated,0,Yes,No,https://www.airbnb.com/rooms/17211057955624048...
4,Athens,New 2BD/2BA Apt • Balcony • Prime & Quiet Loca...,41211,How reviews work,0,Yes,Yes,https://www.airbnb.com/rooms/10970736952302680...


## ⭐ Clean Rating Column

In [ ]:
scraped_df["Rating"] = scraped_df["Rating"].replace("How reviews work", pd.NA)

scraped_df["Rating"] = pd.to_numeric(scraped_df["Rating"], errors="coerce")

scraped_df["Rating"] = scraped_df["Rating"].fillna(0)

scraped_df[["Rating"]].head()

,Rating
0,0.0
1,0.0
2,0.0
3,0.0
4,0.0


In [ ]:
scraped_df.info()

<class 'pandas.DataFrame'>
Index: 396 entries, 0 to 400
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   City            396 non-null    str    
 1   Name            396 non-null    str    
 2   Price           396 non-null    int64  
 3   Rating          396 non-null    float64
 4   Reviews_Count   396 non-null    int64  
 5   Superhost       396 non-null    str    
 6   Guest_Favorite  396 non-null    str    
 7   Link            396 non-null    str    
dtypes: float64(1), int64(2), str(5)
memory usage: 27.8 KB


In [ ]:
scraped_df.to_csv(
    "../02_silver_layer/airbnb_detailed_500_apartments_clean.csv",
    index=False
)

print("Cleaned scraped dataset saved successfully.")

Cleaned scraped dataset saved successfully.
